#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window
from pyspark.sql.functions import trim, col, length

#Read Bronze Table

In [0]:
df = spark.table("demo_project.bronze.erp_loc_a101")

##Sanity checks

In [0]:
df.limit(10).display()

#Silver Transformations

##Renaming Columns

In [0]:
rename_map = {
    "CID": "customer_id",
    "CNTRY": "country"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)


##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


##Customer ID Cleanup

In [0]:
df = df.withColumn("customer_id", F.regexp_replace(col("customer_id"), "-", ""))

##Country Normalization

In [0]:
df = df.withColumn(
    "country",
    F.when(col("country") == "DE", "Germany")
    .when(col("country").isin("US", "USA"), "United States")
    .when((col("country") == "") | col("country").isNull(), "N/A")
    .otherwise(col("country"))

)

##Sanity checks

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("demo_project.silver.erp_customer_location")

##Sanity checks

In [0]:
%sql
select * from demo_project.silver.erp_customer_location limit 10;